# Feature engineering (basic)

Filter numeric features step by step. Print initial feature count, then removed and remaining after each filter.

In [ ]:
from pathlib import Path
import pandas as pd

LEVEL = "paragraph"  # "paragraph" or "sentence" - which data/<LEVEL>/master/ to read and write
master_dir = Path(f"../../data/{LEVEL}/master")
DATA_PATH = master_dir / "master-absolute.csv"
FEATURE_PREFIX = ""  # set "delta_" when using master-pairs-cumulative.csv

SKIP = {
    "article", "revision_group_no", "revision_depth", "base_depth", "edit_depth",
    "label", "text", "base_text", "edit_text", "diff_url",
    "pair_older_depth", "pair_newer_depth",  # pair position info, not text features
}

raw_df = pd.read_csv(DATA_PATH, low_memory=False)
num_df = raw_df.drop(columns=[c for c in raw_df.columns if c in SKIP]).select_dtypes(include="number")
if FEATURE_PREFIX:
    num_df = num_df[[c for c in num_df.columns if c.startswith(FEATURE_PREFIX)]]

initial = list(num_df.columns)
active = list(initial)
print("Total rows:", len(num_df))
print("Initial feature count:", len(active))
print("Initial features:", active)

## Undefined values

For some features the value 0 means **could not be calculated**, not a real 0 (a real value is never 0 for these), e.g.
MDL of a one word paragraph like "Steps" has no dependency arc, vocd needs at least 36 tokens.

1. a feature that is undefined in more than 5% of rows is removed (too many rows have no real value)
2. paragraphs where a kept feature is undefined are saved to `feature-engineering/undefined_paragraphs.csv` -
   every experiment leaves out the pairs that have one of these paragraphs, and they are also left out of the filters below

In [ ]:
# for these features 0 means "could not be calculated" (a real value is never 0)
# maas_ttr and yulek are not here - their 0 is real (no word is repeated)
UNDEFINED_WHEN_ZERO = {
    "mean_mdl": "no dependency arc (one word / only punctuation)",
    "mean_surprisal": "only 1 GPT-2 token, nothing to predict",
    "ttr": "fewer than 2 characters", "msttr": "fewer than 2 characters", "mattr": "fewer than 2 characters",
    "mtld": "fewer than 2 characters", "mtld_ma": "fewer than 2 characters",
    "vocd": "fewer than 36 characters",
    "word_ttr": "fewer than 2 words", "word_msttr": "fewer than 2 words", "word_mattr": "fewer than 2 words",
    "word_mtld": "fewer than 2 words", "word_mtld_ma": "fewer than 2 words",
    "word_vocd": "fewer than 36 words",
}
UNDEFINED_MAX_SHARE = 0.05 # feature undefined in more than 5% of rows is removed

undefined = pd.DataFrame({name: num_df[name] == 0 for name in UNDEFINED_WHEN_ZERO if name in active})
undefined_share = undefined.mean()
print("Undefined share per feature:")
for name, share in undefined_share.sort_values(ascending=False).items():
    print(f"  {name:15} {100 * share:6.2f}%  ({UNDEFINED_WHEN_ZERO[name]})")

#1. remove features that are undefined in too many rows
drop_undefined = sorted(undefined_share[undefined_share > UNDEFINED_MAX_SHARE].index)
print("\nRemoved (undefined in more than 5% of rows):", len(drop_undefined), drop_undefined)
active = [c for c in active if c not in drop_undefined]
num_df = num_df[active]

#2. paragraphs where a kept feature is undefined - saved for the experiments
row_undefined = undefined.drop(columns=drop_undefined).any(axis=1)
key_columns = ["article", "revision_group_no", "revision_depth"]
undefined_paragraphs = raw_df.loc[row_undefined, key_columns + ["text"]].drop_duplicates(key_columns) #middle versions appear twice in master-absolute
out_dir = master_dir / "feature-engineering"
out_dir.mkdir(parents=True, exist_ok=True)
undefined_paragraphs.to_csv(out_dir / "undefined_paragraphs.csv", index=False)
print("Undefined paragraphs:", len(undefined_paragraphs), "(rows in master-absolute:", int(row_undefined.sum()), ") e.g.", undefined_paragraphs["text"].head(5).tolist())

#leave these rows out of the filters below too
num_df = num_df[~row_undefined]
raw_df = raw_df[~row_undefined]
print("Remaining rows:", len(num_df))
print("Remaining:", len(active), active)

In [ ]:
constant_cols = [col for col in active if num_df[col].nunique() == 1]
print("Removed (constant):", len(constant_cols), constant_cols)

drop_const = set(constant_cols)
active = [c for c in active if c not in drop_const]
num_df = num_df[active]
print("Remaining:", len(active), active)

In [ ]:
CORR_THRESH = 0.90
CORR_PREFER_KEEP = {"mean_mdl"}

corr_matrix = num_df.corr().abs()
prefer_keep = {FEATURE_PREFIX + name for name in CORR_PREFER_KEEP}
cols = list(active)
drop_corr = set()

for i, a in enumerate(cols):
    for b in cols[i + 1:]:
        r = corr_matrix.at[a, b]
        if pd.isna(r) or r <= CORR_THRESH:
            continue
        if a in prefer_keep and b not in prefer_keep:
            drop_corr.add(b)
        elif b in prefer_keep and a not in prefer_keep:
            drop_corr.add(a)
        else:
            drop_corr.add(b)

correlated_columns = sorted(drop_corr)
print("Prefer keep:", sorted(prefer_keep))
print("Removed (high correlation):", len(correlated_columns), correlated_columns)

mdl = FEATURE_PREFIX + "mean_mdl"
sen = FEATURE_PREFIX + "avg_sen_len"
if mdl in corr_matrix.columns and sen in corr_matrix.columns:
    r_ms = corr_matrix.at[mdl, sen]
    print(f"\n{mdl} vs {sen}: |r|={r_ms:.3f}")
    print(f"  drop {sen}? {sen in drop_corr}   drop {mdl}? {mdl in drop_corr}")

active = [c for c in active if c not in drop_corr]
num_df = num_df[active]
print("Remaining:", len(active), active)

In [ ]:
# every surprisal feature (gpt2 + srilm) that is still left after the correlation filter
# (not a fixed list - which srilm models survive depends on the data)
surprisal_cols = [c for c in active if c.startswith("mean_surprisal")]
print("Surprisal features:", len(surprisal_cols), surprisal_cols)
print("Pairwise correlation:")
print(num_df[surprisal_cols].corr().round(3))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
TEST_SIZE = 0.2

y = (raw_df["label"].astype(str) == "edit").astype(int).to_numpy()
articles = raw_df["article"].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(gss.split(raw_df, y, groups=articles))

results = []
for cols in [[c] for c in surprisal_cols] + [surprisal_cols]:
    name = cols[0] if len(cols) == 1 else "all_surprisal"
    X = num_df[cols].fillna(0.0).to_numpy()
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
    ])
    pipe.fit(X[train_idx], y[train_idx])
    pred = pipe.predict(X[test_idx])
    proba = pipe.predict_proba(X[test_idx])[:, 1]
    acc = accuracy_score(y[test_idx], pred)
    auc = roc_auc_score(y[test_idx], proba)
    results.append({"name": name, "n_features": len(cols), "accuracy": acc, "auc": auc})
    print(f"{name}: acc={acc:.4f}  auc={auc:.4f}  n={len(cols)}")

best_single = max((r for r in results if r["n_features"] == 1), key=lambda r: r["auc"])
all_five = next(r for r in results if r["name"] == "all_surprisal")
gain_auc = all_five["auc"] - best_single["auc"]
gain_acc = all_five["accuracy"] - best_single["accuracy"]
print(f"\nBest single: {best_single['name']}  acc={best_single['accuracy']:.4f}  auc={best_single['auc']:.4f}")
print(f"All together gain vs best single: acc={gain_acc:+.4f}  auc={gain_auc:+.4f}")
if abs(gain_auc) <= 0.01 and abs(gain_acc) <= 0.01:
    print(f"Recommendation: keep only {best_single['name']}; discard the rest.")
else:
    print("Recommendation: keep all surprisal features (all-together gain is meaningful).")

In [ ]:

# Apply surprisal keep-one from the ablation cell above.
# GPT-2 surprisal is kept (same as the earlier experiments); the ablation above is only for reference
keep_surprisal = "mean_surprisal"
drop_surprisal = [c for c in surprisal_cols if c != keep_surprisal]
active = [c for c in active if c not in set(drop_surprisal)]
num_df = num_df[[c for c in active if c in num_df.columns]]
print("Surprisal keep:", keep_surprisal)
print("Surprisal drop:", drop_surprisal)

out_dir = master_dir / "feature-engineering"
out_dir.mkdir(parents=True, exist_ok=True)

dropped = [c for c in initial if c not in set(active)]
pd.DataFrame({"feature": dropped}).to_csv(out_dir / "dropped_features.csv", index=False)
pd.DataFrame({"feature": active}).to_csv(out_dir / "final_features.csv", index=False)

print("Dropped:", len(dropped), dropped)
print("Final:", len(active), active)